<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/01_ingesta/03_ENARES_2024_CRS04_identificar_modulo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENARES 2024 CRS04 ML Pipeline

## Notebook 3 — Identificación CRS01–CRS04 y confirmación CRS04

### Stage 1 — Data Ingestion

Este notebook identifica los módulos **CRS01, CRS02, CRS03 y CRS04** de ENARES 2024 a partir de los archivos `.sav` descargados desde INEI en formato **SPSS ZIP**.

El objetivo de este notebook, para el **Issue #5**, es confirmar qué archivos pertenecen a cada CRS y documentar la identificación de **CRS04** usando evidencia reproducible:

- nombre de archivo;
- número de filas;
- número de columnas;
- metadata SPSS.

---

# Notebook Information

| Field | Value |
|---|---|
| **Notebook name** | `03_ENARES_2024_CRS04_identificar_modulo.ipynb` |
| **Stage** | Stage 1 — Data Ingestion |
| **Role** | Computer Science Lead — Data Engineering |
| **Author** | Ana Cordero Ricaldi |
| **Account used** | `anacordero.001@gmail.com` |
| **Project scope** | ENARES 2024 CRS04 ML Pipeline |
| **Input format** | SPSS `.sav` extracted from official INEI SPSS ZIP packages |
| **Modules identified** | CRS01, CRS02, CRS03, CRS04 |
| **Main target module** | CRS04 |
| **Issue linked** | Issue #5 |
| **Script version** | `stage1-crs-identification-v1.0` |
| **Date executed** | `[AUTO-GENERATED AT RUNTIME]` |

---

# OBJECTIVE

Identify and document which ENARES 2024 `.sav` files correspond to CRS01, CRS02, CRS03 and CRS04.

The notebook provides reproducible evidence to confirm the CRS04 module by inspecting structural information and preserved SPSS metadata.

---

# FUNCTION OF THIS NOTEBOOK

This notebook is responsible only for:

- scanning extracted `.sav` files from Stage 1 ingestion;
- reading SPSS metadata without modifying the raw files;
- identifying candidate CRS modules;
- comparing CRS01, CRS02, CRS03 and CRS04 structures;
- recording file names, row counts and column counts;
- preserving variable labels and value label information;
- producing identification evidence for CRS04;
- generating a markdown report for supervisory review.

---

# METHODOLOGICAL DECISION

CRS module identification is performed using the original `.sav` files because SPSS format preserves metadata needed for reproducible interpretation.

The identification process relies on:

- file names;
- variable names;
- variable labels;
- value labels;
- row counts;
- column counts;
- questionnaire/module structure.

This avoids relying only on informal file naming and provides auditable evidence for the CRS04 selection.

---

# STRICT LIMITS OF THIS NOTEBOOK

This notebook strictly **DOES NOT** perform:

- data cleaning;
- recoding;
- merging;
- feature engineering;
- statistical analysis;
- visualisation;
- modelling;
- BigQuery loading;
- changes to raw `.sav` files.

Its function is exclusively **identification, traceability and initial structural review**.

---

# EXPECTED INPUTS

```text
01BasesDatosPrimarias/
└── extraidos/
    └── *.sav
```

---

# EXPECTED OUTPUTS

```text
05Resultados/logs/
├── ENARES_2024_CRS04_variables_stage1.csv
├── ENARES_2024_CRS04_value_labels_stage1.csv
├── ENARES_2024_CRS04_missing_codes_stage1.csv
└── ENARES_2024_CRS04_validacion_stage1.csv

04CuestionariosInformes/reportes/
└── ENARES_2024_CRS_identificacion_modulos.md
```

---

# REPRODUCIBILITY REQUIREMENT

The notebook must be executable from the outputs generated by Stage 1 ingestion and must consistently reproduce:

- the CRS01–CRS04 identification table;
- the CRS04 file confirmation;
- row and column counts;
- extracted SPSS metadata summaries;
- the CRS identification report.

All outputs must be traceable to the original `.sav` files extracted from official INEI SPSS ZIP packages.

---

# CURRENT STATUS

Notebook prepared for:

- scanning extracted SPSS files;
- identifying CRS module candidates;
- confirming CRS04;
- extracting structural metadata;
- producing Issue #5 evidence;
- generating the CRS module identification report.

In [ ]:
!pip install pyreadstat pandas

In [ ]:
import os
import pandas as pd
import pyreadstat
from google.colab import drive
from datetime import datetime

In [ ]:
drive.mount('/content/drive')

In [ ]:
ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = os.path.join(ROOT, "01BasesDatosPrimarias")

REPORT_DIR = os.path.join(ROOT, "04CuestionariosInformes/reportes")
LOG_DIR = os.path.join(ROOT, "05Resultados/logs")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
sav_files = []

for root, _, files in os.walk(RAW_DIR):
    for f in files:
        if f.endswith(".sav"):
            sav_files.append(os.path.join(root, f))



## CRS Detection Rule
CRS detection is primarily filename-based because the official INEI SPSS files include the CRS identifier in the file name. Row counts, column counts and SPSS metadata are used as validation evidence, not as the primary classification rule.

Keyword scoring is used only as supportive thematic evidence. It is not used as the primary CRS selection rule because other CRS modules may also contain violence-related labels.
"""

In [ ]:
def extract_metadata(file_path):

    df, meta = pyreadstat.read_sav(file_path)

    module_id = os.path.basename(os.path.dirname(file_path))

    return {
        "file": file_path,
        "module": module_id,
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "columns": list(df.columns),
        "column_labels": meta.column_names_to_labels,
        "value_labels": meta.variable_value_labels,
        "missing_ranges": meta.missing_ranges
    }

In [ ]:
metadata_list = []

for f in sav_files:
    try:
        metadata_list.append(extract_metadata(f))
        print("Processed:", f)
    except Exception as e:
        print("Failed:", f, e)

In [ ]:
crs_detection_rows = []

for m in metadata_list:

    file_name = os.path.basename(m["file"]).upper()

    detected_crs = "UNKNOWN"

    if "CRS01" in file_name:
        detected_crs = "CRS01"

    elif "CRS02" in file_name:
        detected_crs = "CRS02"

    elif "CRS03" in file_name:
        detected_crs = "CRS03"

    elif "CRS04" in file_name:
        detected_crs = "CRS04"

    crs_detection_rows.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "detected_crs": detected_crs,
        "n_rows": m["n_rows"],
        "n_columns": m["n_columns"]
    })

df_crs_detection = pd.DataFrame(crs_detection_rows)

df_crs_detection = df_crs_detection.sort_values(by=["module_id", "sav_file"]).reset_index(drop=True)

### 2. Documentar claramente los archivos `.sav` procesados

In [ ]:
display(df_crs_detection)

expected_n_modules = 22
processed_n_modules = len(df_crs_detection)

if processed_n_modules == expected_n_modules:
    print("PASS: All 22 expected ENARES 2024 .sav modules were processed.")
else:
    print(f"WARNING: Expected {expected_n_modules} modules, but processed {processed_n_modules}.")

unknown_modules = df_crs_detection[df_crs_detection["detected_crs"] == "UNKNOWN"]
if unknown_modules.empty:
    print("PASS: No UNKNOWN CRS modules detected.")
else:
    print("WARNING: Some modules could not be classified:")
    display(unknown_modules)

df_crs_summary = (
    df_crs_detection
    .groupby("detected_crs")
    .agg(
        n_files=("sav_file", "count"),
        min_rows=("n_rows", "min"),
        max_rows=("n_rows", "max"),
        total_columns=("n_columns", "sum")
    )
    .reset_index()
)
print("\n--- CRS Detection Summary ---")
display(df_crs_summary)

Keyword scoring was used as supportive thematic evidence only. It was not used as the primary CRS selection rule because other CRS modules may also contain violence-related labels. CRS04 selection was based primarily on filename evidence and validated through row structure and SPSS metadata.

*Note: The following table is exploratory only. It is not used to select CRS04 files. CRS04 files are selected by official filename pattern.*

In [ ]:
keywords = [
     # violence core
    "violencia", "violento", "golpe", "golpear", "agresión", "agresion",
    "insulto", "humillación", "amenaza", "maltrato", "abuso",

    # sexual violence
    "sexual", "tocamiento", "acoso", "violación", "violacion",

    # school context
    "escuela", "colegio", "aula", "profesor", "clase", "institución educativa",

    # family context
    "familia", "hogar", "padre", "madre", "hermano", "tutor", "casa",

    # social support / relationships
    "apoyo", "redes", "amigos", "amistades", "relación", "relaciones",

    # gender roles / perception
    "género", "genero", "roles", "actitudes", "creencias",

    # adolescent context (soft signals)
    "adolescente", "estudiante", "joven"
]

crs04_results = []

for m in metadata_list:

    file_name = m["file"].upper()
    columns = m["columns"]
    labels = m["column_labels"]

    # ---- 1. filename signal ----
    filename_score = 1 if "CRS04" in file_name else 0

    # ---- 2. label-based semantic score ----
    label_text = " ".join(
        [str(labels.get(col, "")) for col in columns]
    ).lower()

    keyword_score = sum(label_text.count(k) for k in keywords)

    # ---- 3. structure signal ----
    n_rows = m["n_rows"]
    n_cols = m["n_columns"]

    structure_score = 1 if n_rows == 18807 else 0  # CRS04 expected 18807

    # ---- FINAL SCORE ----
    total_score = filename_score + keyword_score + structure_score

    crs04_results.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "n_rows": n_rows,
        "n_columns": n_cols,
        "filename_score": filename_score,
        "keyword_score": keyword_score,
        "structure_score": structure_score,
        "total_score": total_score
    })

df_crs04_scores = pd.DataFrame(crs04_results)
df_crs04_scores.sort_values("total_score", ascending=False)

In [ ]:
df_crs04_candidates = df_crs04_scores[
    df_crs04_scores["sav_file"].str.contains("CRS04", case=False)
]
df_crs04_candidates = df_crs04_candidates.sort_values(
    "total_score",
    ascending=False
)
display(df_crs04_candidates)

df_crs04_confirmed = df_crs_detection[df_crs_detection["detected_crs"] == "CRS04"].copy()
df_crs04_confirmed = df_crs04_confirmed.sort_values(by=["module_id", "sav_file"]).reset_index(drop=True)

display(df_crs04_confirmed)

expected_crs04_files = 4
actual_crs04_files = df_crs04_confirmed.shape[0]

if actual_crs04_files == expected_crs04_files:
    print("PASS: Four CRS04 files were identified.")
else:
    print(f"WARNING: Expected 4 CRS04 files, but found {actual_crs04_files}.")

expected_crs04_rows = 18807
invalid_crs04_rows = df_crs04_confirmed[df_crs04_confirmed["n_rows"] != expected_crs04_rows]

if invalid_crs04_rows.empty:
    print("PASS: All CRS04 files have 18,807 rows.")
else:
    print("WARNING: Some CRS04 files do not have the expected number of rows:")
    display(invalid_crs04_rows)

CRS04_FILES = df_crs04_confirmed["sav_file"].tolist()
CRS04_MODULES = df_crs04_confirmed["module_id"].tolist()

print("CRS04 modules found:", CRS04_MODULES)
print("Number of CRS04 files:", len(CRS04_FILES))

In [ ]:
# ============================================================
# STAGE 1 PROGRAM ASSERTIONS - CRS04 STRUCTURAL INTEGRITY
# Issue #10
# ============================================================
EXPECTED_CRS04_FILES = 4
EXPECTED_CRS04_ROWS = 18807
MIN_EXPECTED_COLUMNS = 1

assert len(df_crs04_confirmed) == EXPECTED_CRS04_FILES, (
    f"Expected {EXPECTED_CRS04_FILES} CRS04 files, found {len(df_crs04_confirmed)}"
)

assert (df_crs04_confirmed["n_rows"] == EXPECTED_CRS04_ROWS).all(), (
    "All CRS04 files must have 18,807 rows at Stage 1."
)

assert (df_crs04_confirmed["n_columns"] >= MIN_EXPECTED_COLUMNS).all(), (
    "Every CRS04 file must contain at least one readable column."
)

assert df_crs04_confirmed["sav_file"].str.contains("CRS04", case=False, na=False).all(), (
    "All confirmed CRS04 files must contain CRS04 in the official filename."
)

assert df_crs04_confirmed["module_id"].nunique() == EXPECTED_CRS04_FILES, (
    "Each CRS04 file must belong to a distinct INEI module folder."
)

assertion_results = {
    "crs04_file_count_pass": True,
    "crs04_expected_rows_pass": True,
    "crs04_min_columns_pass": True,
    "crs04_filename_rule_pass": True,
    "crs04_distinct_modules_pass": True,
    "expected_crs04_rows": EXPECTED_CRS04_ROWS,
    "observed_crs04_files": int(len(df_crs04_confirmed)),
    "observed_crs04_modules": int(df_crs04_confirmed["module_id"].nunique()),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}

assertion_results_path = os.path.join(LOG_DIR, "ENARES_2024_STAGE1_assertion_results.csv")
pd.DataFrame([assertion_results]).to_csv(assertion_results_path, index=False)

print("PASS: CRS04 structural assertions completed successfully.")
print("Assertion results saved:", assertion_results_path)
print(assertion_results)

In [ ]:
df_vars = []
df_values = []
df_missing = []

for m in metadata_list:

    if m["file"] not in CRS04_FILES:
        continue

    for var in m["columns"]:
        df_vars.append({
            "variable_name": var,
            "variable_label": m["column_labels"].get(var),
            "variable_type": "unknown",
            "source_module": m["module"],
            "source_file": m["file"]
        })

    for var, labels in m["value_labels"].items():
        if isinstance(labels, dict):
            for value, label in labels.items():
                df_values.append({
                    "variable_name": var,
                    "value": value,
                    "value_label": label,
                    "source_module": m["module"],
                    "source_file": m["file"]
                })

        for var, miss in m["missing_ranges"].items():
            df_missing.append({
                "variable_name": var,
                "missing_code": str(miss),
                "missing_label_or_type": "SPSS_missing",
                "source_module": m["module"],
                "source_file": m["file"],
                "notes": "Extracted via pyreadstat missing_ranges"
            })

df_vars = pd.DataFrame(df_vars)
df_values = pd.DataFrame(df_values)
df_missing = pd.DataFrame(df_missing)



In [ ]:
df_vars.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_variables_stage1.csv"),
    index=False
)

df_values.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_value_labels_stage1.csv"),
    index=False
)

df_missing.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_missing_codes_stage1.csv"),
    index=False
)


In [ ]:
validation_rows = []

for m in metadata_list:
    if m["file"] not in CRS04_FILES:
        continue

    cols = m["columns"]
    text = " ".join(cols).lower()

    age_vars = [v for v in cols if "edad" in v.lower()]
    sex_vars = [v for v in cols if "sexo" in v.lower()]

    # disability range: C4P130_1 to C4P130_6
    disability_vars = [
        v for v in cols
        if v.lower().startswith("c4p130_")
    ]

    weight_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["factor_alumnos"])
    ]

    strata_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["ccdd"])
    ]

    cluster_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["id"])
    ]

    validation_rows.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "n_rows": m["n_rows"],
        "n_columns": m["n_columns"],

        # now EVIDENCE instead of boolean
        "possible_age_variables": age_vars,
        "possible_sex_variables": sex_vars,
        "possible_disability_variables": disability_vars,
        "possible_weight_variables": weight_vars,
        "possible_strata_variables": strata_vars,
        "possible_cluster_variables": cluster_vars,

        "notes":(
            "CRS04 validation with explicit variable evidence extraction. "
            "Supervisor-confirmed Stage 1 interpretation: "
            "CCDD = Departamento / strata; "
            "ID = Colegio / conglomerado / UPM; "
            "FACTOR_ALUMNOS = survey weight; "
            "C4P130_1-C4P130_6 = disability variables. "
            "Age and sex variables were searched using both variable names and SPSS labels."
        )
    })

df_validation = pd.DataFrame(validation_rows)

df_validation.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_validacion_stage1.csv"),
    index=False
)

expected_outputs = [
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_variables_stage1.csv"),
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_value_labels_stage1.csv"),
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_missing_codes_stage1.csv"),
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_validacion_stage1.csv"),
]

print("\n--- OUTPUT VALIDATION ---")
for path in expected_outputs:
    if os.path.exists(path):
        print(f"PASS: Output exists: {path}")
    else:
        print(f"WARNING: Missing output: {path}")

In [ ]:
report_path = os.path.join(REPORT_DIR, "ENARES_2024_CRS_identificacion_modulos.md")

crs04_markdown_rows = ""
for _, row in df_crs04_confirmed.iterrows():
    base_file = os.path.basename(row['sav_file'])
    crs04_markdown_rows += f"| {row['module_id']} | {base_file} | {row['n_rows']} | {row['n_columns']} |\n"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("""# ENARES 2024 — CRS Module Identification Report

## Objective
Identify CRS01, CRS02, CRS03 and CRS04 modules from the downloaded SPSS `.sav` files.

## Stage 1 Scope
This report belongs to Stage 1 — Data Ingestion. No cleaning, recoding, merging, statistical analysis, graphics or modelling was performed.

## Input Files
A total of 22 ENARES 2024 `.sav` files were processed, covering modules `976-Modulo1941` to `976-Modulo1962`.

## CRS Detection Method
- Filename-based CRS detection (Primary rule).
- Row and column structure review.
- SPSS metadata thematic review.
- Keyword scoring used only as supportive evidence.

## Use of Keyword Scoring
Keyword scoring was used only as supportive thematic evidence to review whether CRS04 files contained labels consistent with the expected questionnaire theme. It was not used as the primary CRS selection rule. The primary rule was filename-based CRS detection, because the official INEI `.sav` files include the CRS identifier in the file name.

## CRS04 Initial Validation and SPSS Metadata
The four CRS04 `.sav` files were read using `pyreadstat`.

The following metadata were extracted:
- variable names;
- variable labels;
- value labels;
- missing ranges / missing codes where available;
- row and column counts.

The following Stage 1 structural variables were documented:
- `CCDD` = Departamento / strata;
- `ID` = Colegio / conglomerado / UPM;
- `FACTOR_ALUMNOS` = survey weight;
- `C4P130_1-C4P130_6` = disability variables.

## CRS Files Summary
| Detected CRS | File Count | Min Rows | Max Rows | Total Columns |
|---|---|---|---|---|
""")
    for _, row in df_crs_summary.iterrows():
        f.write(f"| {row['detected_crs']} | {row['n_files']} | {row['min_rows']} | {row['max_rows']} | {row['total_columns']} |\n")

    f.write(f"""
## CRS04 Files Identified
The following four CRS04 files were identified from the ENARES 2024 SPSS `.sav` files:

| module_id | file | rows | columns |
|---|---|---|---|
{crs04_markdown_rows}

## CRS04 Selection Evidence
- Filename contains CRS04.
- Four CRS04 files were identified.
- All four CRS04 files have 18,807 rows.
- SPSS metadata was reviewed.
- CRS04 contains the expected disability variables `C4P130_1` to `C4P130_6`.

## Outputs Generated
- `04CuestionariosInformes/reportes/ENARES_2024_CRS_identificacion_modulos.md` (Primary)
- `05Resultados/logs/ENARES_2024_CRS04_variables_stage1.csv` (Supporting)
- `05Resultados/logs/ENARES_2024_CRS04_value_labels_stage1.csv` (Supporting)
- `05Resultados/logs/ENARES_2024_CRS04_validacion_stage1.csv` (Supporting)
- `05Resultados/logs/ENARES_2024_CRS04_missing_codes_stage1.csv` (Supporting)

## Linked outputs for Issue #5 and #6
**Primary output:**
- `04CuestionariosInformes/reportes/ENARES_2024_CRS_identificacion_modulos.md`

**Supporting outputs:**
- `05Resultados/logs/ENARES_2024_CRS04_validacion_stage1.csv`
- `05Resultados/logs/ENARES_2024_CRS04_variables_stage1.csv`
- `05Resultados/logs/ENARES_2024_CRS04_value_labels_stage1.csv`
- `05Resultados/logs/ENARES_2024_CRS04_missing_codes_stage1.csv`

## Limitations
- Keyword scoring is supportive only.
- This notebook does not construct analytical variables.
- This notebook does not merge CRS04 files.
- This notebook does not clean or recode data.
- Final survey design and analytical variables are validated in later stages.

## Stage 1 Decision
CRS01, CRS02, CRS03 and CRS04 modules were identified successfully. CRS04 identification is accepted for Stage 1 based on:
- official filename evidence;
- expected CRS04 row structure (18,807 rows);
- SPSS metadata review;
- confirmation of the four CRS04 source files.

These files will be used as input for CRS04 metadata extraction, Stage 2 BigQuery loading, and Stage 3 ETL validation. This decision does not imply that analytical variables have been created or validated. It only confirms CRS04 source file identification.
""")

print("Report saved:", report_path)

# --- MARKDOWN CELL ---
"""
## Stage 1 Decision — CRS04 Identification

The four CRS04 files were successfully identified from the ENARES 2024 SPSS `.sav` files.
CRS04 identification is accepted for Stage 1 based on:
- official filename evidence;
- expected CRS04 row structure;
- SPSS metadata review;
- confirmation of the four CRS04 source files.

No cleaning, recoding, merging, statistical analysis, graphics or modelling was performed.
"""

print("\n--- STAGE 1 DECISION ---")
print("STAGE 1 DECISION: CRS04 identification accepted.")
print("Evidence: filename pattern, row structure (18,807 rows), SPSS metadata review.")
print("No cleaning, recoding, merging, statistical analysis or modelling was performed.")